In [1]:

import numpy as np


# ==========================================
# 1. ACP DIRECTE
# ==========================================

def acp_directe_fit(X_train, q):
    """
    ACP directe:
    - Centrage
    - Covariance
    - Diagonalisation
    - Tri des valeurs propres
    - Selection des q composantes
    """

    n, d = X_train.shape

    # 1. Calcul de la moyenne
    mean = np.mean(X_train, axis=0)

    # 2. Centrage
    X_centered = X_train - mean

    # 3. Matrice de covariance
    S = (X_centered.T @ X_centered) / (n - 1)

    # 4. Diagonalisation
    eigenvalues, eigenvectors = np.linalg.eigh(S)

    # 5. Tri decroissant
    indices = np.argsort(eigenvalues)[::-1]

    eigenvalues = eigenvalues[indices]
    eigenvectors = eigenvectors[:, indices]

    # 6. Selection des q composantes
    V = eigenvectors[:, :q]
    selected_eigenvalues = eigenvalues[:q]

    return {
        "mean": mean,
        "components": V,
        "eigenvalues": selected_eigenvalues,
        "all_eigenvalues": eigenvalues
    }


def acp_directe_transform(X, model):
    """
    Projection des donnees vers q dimensions.
    """

    X_centered = X - model["mean"]

    Z = X_centered @ model["components"]

    return Z


def acp_directe_reconstruction(Z, model):
    """
    Reconstruction des donnees dans l'espace initial.
    """

    X_reconstructed = (
        Z @ model["components"].T + model["mean"]
    )

    return X_reconstructed


In [2]:

# ==========================================
# 2. ACP ITERATIVE
# ==========================================

def acp_iterative_fit(
    X_train,
    q,
    max_iter=1000,
    tol=1e-8,
    seed=42
):
    """
    ACP iterative par iteration de sous-espace.

    On cherche les q vecteurs propres principaux
    sans diagonaliser directement toute la covariance.
    """

    n, d = X_train.shape

    # 1. Centrage
    mean = np.mean(X_train, axis=0)
    X_centered = X_train - mean

    # 2. Covariance
    S = (X_centered.T @ X_centered) / (n - 1)

    # 3. Initialisation aleatoire
    rng = np.random.default_rng(seed)
    Q = rng.normal(size=(d, q))

    # Orthogonalisation initiale
    Q, _ = np.linalg.qr(Q)

    # 4. Iteration de sous-espace
    for iteration in range(max_iter):

        Y = S @ Q

        Q_new, _ = np.linalg.qr(Y)

        # Mesure de convergence entre les sous-espaces
        difference = np.linalg.norm(
            Q_new @ Q_new.T - Q @ Q.T,
            ord="fro"
        )

        Q = Q_new

        if difference < tol:
            break

    # 5. Diagonalisation dans le sous-espace trouve
    B = Q.T @ S @ Q

    values, vectors = np.linalg.eigh(B)

    # Trier les valeurs propres
    indices = np.argsort(values)[::-1]

    values = values[indices]
    vectors = vectors[:, indices]

    # Vecteurs propres approches
    components = Q @ vectors

    return {
        "mean": mean,
        "components": components,
        "eigenvalues": values,
        "n_iterations": iteration + 1,
        "converged": difference < tol
    }


def acp_iterative_transform(X, model):
    """
    Transformation avec les composantes iteratives.
    """

    X_centered = X - model["mean"]

    Z = X_centered @ model["components"]

    return Z


def acp_iterative_reconstruction(Z, model):
    """
    Reconstruction apres ACP iterative.
    """

    X_reconstructed = (
        Z @ model["components"].T + model["mean"]
    )

    return X_reconstructed


In [4]:

# ==========================================
# 3. PROJECTION GAUSSIENNE
# ==========================================

def projection_gaussienne_fit(d, q, seed=42):
    """
    Generation de la matrice aleatoire R.
    """

    rng = np.random.default_rng(seed)

    R = rng.normal(
        loc=0,
        scale=1 / np.sqrt(q),
        size=(d, q)
    )

    return {
        "R": R,
        "q": q,
        "seed": seed
    }


def projection_gaussienne_transform(X, model):
    """
    Projection des donnees vers q dimensions.
    """

    Z = X @ model["R"]

    return Z


def repetitions_gaussiennes(X, q, seeds):
    """
    Repeter les projections avec plusieurs graines.
    """

    resultats = {}

    for seed in seeds:

        model = projection_gaussienne_fit(
            d=X.shape[1],
            q=q,
            seed=seed
        )

        Z = projection_gaussienne_transform(X, model)

        resultats[seed] = Z

    return resultats
